In [ ]:
# Import libraries

import os
import json
import time
import random

import numpy as np
import torch
import matplotlib.pyplot as plt

import monai
from monai.data import Dataset, DataLoader, decollate_batch
from monai.networks.nets import UNet
from monai.losses import DiceFocalLoss
from monai.metrics import DiceMetric
from monai.transforms import (
    LoadImaged,
    EnsureChannelFirstd,
    ConcatItemsd,
    NormalizeIntensityd,
    RandCropByPosNegLabeld,
    MapTransform,
    AsDiscrete,
    Compose,
)
from monai.inferers import sliding_window_inference

import wandb
from dotenv import load_dotenv

load_dotenv()

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"MONAI version: {monai.__version__}")
print(f"Using device: {device}")

In [ ]:
# Config & Paths
# Project paths and settings

BASE_DIR = r"D:\Deep_Projects\brain-tumor-segmentation-3d\repo"

PATHS = {
    "training_dir": os.path.join(BASE_DIR, "data", "brats2020", "BraTS2020_TrainingData", "MICCAI_BraTS2020_TrainingData"),
    "configs": os.path.join(BASE_DIR, "configs"),
    "models": os.path.join(BASE_DIR, "models", "improved"),
    "figures": os.path.join(BASE_DIR, "results", "figures"),
    "metrics": os.path.join(BASE_DIR, "results", "metrics"),
}

for key in ["models", "figures", "metrics"]:
    os.makedirs(PATHS[key], exist_ok=True)

print("Paths configured:")
for name, path in PATHS.items():
    status = "OK" if os.path.exists(path) else "missing"
    print(f"  [{status}] {name:14s} -> {path}")

In [ ]:
# Model, loss function, and label remapping
# Same architecture as the baseline (notebook 03) — testing more epochs, not a different model

class RemapLabeld(MapTransform):
    def __call__(self, data):
        d = dict(data)
        for key in self.keys:
            d[key][d[key] == 4] = 3
        return d


model = UNet(
    spatial_dims=3,
    in_channels=4,
    out_channels=4,
    channels=(16, 32, 64, 128, 256),
    strides=(2, 2, 2, 2),
    num_res_units=2,
).to(device)

loss_function = DiceFocalLoss(
    to_onehot_y=True,
    softmax=True,
    lambda_dice=0.5,
    lambda_focal=0.5,
)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model built: 3D U-Net ({n_params:,} parameters)")
print("Loss function: DiceFocalLoss (0.5 * Dice + 0.5 * Focal)")

In [ ]:
# Load dataset split and build the data pipeline

split_path = os.path.join(PATHS["configs"], "dataset_split.json")
with open(split_path, "r") as f:
    split_dict = json.load(f)

train_patients = split_dict["train"]
val_patients = split_dict["val"]

modalities = ["t1", "t1ce", "t2", "flair"]
patch_size = (96, 96, 96)

def build_data_dicts(patient_list, training_dir):
    data_dicts = []
    for pid in patient_list:
        patient_dir = os.path.join(training_dir, pid)
        entry = {mod: os.path.join(patient_dir, f"{pid}_{mod}.nii") for mod in modalities}
        entry["label"] = os.path.join(patient_dir, f"{pid}_seg.nii")
        entry["patient_id"] = pid
        data_dicts.append(entry)
    return data_dicts

train_dicts = build_data_dicts(train_patients, PATHS["training_dir"])
val_dicts = build_data_dicts(val_patients, PATHS["training_dir"])

train_transforms = Compose([
    LoadImaged(keys=modalities + ["label"]),
    EnsureChannelFirstd(keys=modalities + ["label"]),
    RemapLabeld(keys=["label"]),
    ConcatItemsd(keys=modalities, name="image"),
    NormalizeIntensityd(keys="image", nonzero=True, channel_wise=True),
    RandCropByPosNegLabeld(
        keys=["image", "label"],
        label_key="label",
        spatial_size=patch_size,
        pos=1,
        neg=1,
        num_samples=2,
        image_key="image",
        image_threshold=0,
    ),
])

val_transforms = Compose([
    LoadImaged(keys=modalities + ["label"]),
    EnsureChannelFirstd(keys=modalities + ["label"]),
    RemapLabeld(keys=["label"]),
    ConcatItemsd(keys=modalities, name="image"),
    NormalizeIntensityd(keys="image", nonzero=True, channel_wise=True),
])

train_ds = Dataset(data=train_dicts, transform=train_transforms)
val_ds = Dataset(data=val_dicts, transform=val_transforms)

train_loader = DataLoader(train_ds, batch_size=1, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=1, num_workers=0)

print(f"Train dataset: {len(train_ds)} patients")
print(f"Val dataset: {len(val_ds)} patients")

In [ ]:
# Checkpoint / Resume setup
# Saves full training state after every epoch so a long run can survive interruptions

checkpoint_path = os.path.join(PATHS["models"], "checkpoint.pth")
best_model_path = os.path.join(PATHS["models"], "best_model.pth")

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)

num_epochs = 45
val_interval = 5

start_epoch = 0
best_dice = -1
best_epoch = -1

if os.path.exists(checkpoint_path):
    print(f"Found existing checkpoint -> resuming training")
    checkpoint = torch.load(checkpoint_path)
    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    start_epoch = checkpoint["epoch"] + 1
    best_dice = checkpoint["best_dice"]
    best_epoch = checkpoint["best_epoch"]
    print(f"Resuming from epoch {start_epoch}, best Dice so far: {best_dice:.4f}")
else:
    print("No checkpoint found -> starting training from scratch")

print(f"Will train epochs {start_epoch} to {num_epochs - 1}")

In [ ]:
# W&B setup

wandb_key = os.environ.get("WANDB_API_KEY")
if wandb_key:
    wandb.login(key=wandb_key, relogin=True)
    print("W&B login successful")
else:
    print("WARNING: WANDB_API_KEY not found in environment")

wandb.init(
    project="brain-tumor-segmentation-3d",
    name="improved-3dunet-45epochs",
    config={
        "architecture": "3D U-Net (MONAI)",
        "patch_size": patch_size,
        "batch_size": 1,
        "patches_per_volume": 2,
        "lr": 1e-4,
        "num_epochs": num_epochs,
        "loss": "DiceFocalLoss",
        "note": "extended training vs baseline (15 epochs, Dice 0.657)",
    },
    resume="allow",
)

print(f"W&B run initialized: improved-3dunet-45epochs")

In [ ]:
# Main training loop with checkpoint saving after every epoch

dice_metric = DiceMetric(include_background=False, reduction="mean")
post_pred = AsDiscrete(argmax=True, to_onehot=4)
post_label = AsDiscrete(to_onehot=4)

for epoch in range(start_epoch, num_epochs):
    model.train()
    epoch_loss = 0
    start_time = time.time()

    for batch_data in train_loader:
        inputs = batch_data["image"].to(device)
        labels = batch_data["label"].to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = loss_function(outputs, labels)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    epoch_loss /= len(train_loader)
    epoch_time = time.time() - start_time

    wandb.log({"train_loss": epoch_loss, "epoch": epoch, "epoch_time_sec": epoch_time})
    print(f"Epoch {epoch+1}/{num_epochs} — loss: {epoch_loss:.4f} — time: {epoch_time:.1f}s")

    # Validation
    if (epoch + 1) % val_interval == 0:
        model.eval()
        dice_metric.reset()

        with torch.no_grad():
            for val_data in val_loader:
                val_inputs = val_data["image"].to(device)
                val_labels = val_data["label"].to(device)

                val_outputs = sliding_window_inference(val_inputs, patch_size, sw_batch_size=1, predictor=model)

                val_outputs_list = decollate_batch(val_outputs)
                val_labels_list = decollate_batch(val_labels)

                val_outputs_convert = [post_pred(x) for x in val_outputs_list]
                val_labels_convert = [post_label(x) for x in val_labels_list]

                dice_metric(y_pred=val_outputs_convert, y=val_labels_convert)

        mean_dice = dice_metric.aggregate().item()
        wandb.log({"val_dice": mean_dice, "epoch": epoch})
        print(f"  -> Validation Dice: {mean_dice:.4f}")

        if mean_dice > best_dice:
            best_dice = mean_dice
            best_epoch = epoch
            torch.save(model.state_dict(), best_model_path)
            print(f"  -> New best model saved (Dice: {best_dice:.4f})")

    # Save full checkpoint after every epoch (for resume)
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "best_dice": best_dice,
        "best_epoch": best_epoch,
    }, checkpoint_path)

print(f"\nTraining complete. Best Dice: {best_dice:.4f} at epoch {best_epoch+1}")

In [ ]:
# Compare baseline (notebook 03) vs improved (this notebook) training curves

baseline_dice = {5: 0.5792, 10: 0.6120, 15: 0.6570}
improved_dice = {5: 0.4831, 10: 0.6309, 15: 0.5539, 20: 0.6640, 25: 0.6794, 30: 0.6736, 35: 0.6785, 40: 0.6863, 45: 0.6842}

fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(list(baseline_dice.keys()), list(baseline_dice.values()), marker="o", label="Baseline (15 epochs)", color="orchid")
ax.plot(list(improved_dice.keys()), list(improved_dice.values()), marker="o", label="Improved (45 epochs)", color="darkorange")
ax.axhline(0.6863, color="gray", linestyle="--", linewidth=1, label="Best overall: 0.6863")

ax.set_xlabel("Epoch")
ax.set_ylabel("Validation Dice")
ax.set_title("Baseline vs Improved: Validation Dice Progression")
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
save_path = os.path.join(PATHS["figures"], "baseline_vs_improved_comparison.png")
plt.savefig(save_path, dpi=120, bbox_inches="tight")
plt.show()
print(f"Saved -> {save_path}")

In [ ]:
# Summary

print("NOTEBOOK 04b COMPLETE")
print("=" * 50)
print(f"Extended training: 45 epochs (vs baseline's 15)")
print(f"Checkpoint/resume: implemented and tested (survived a real power/network outage)")
print()
print(f"Baseline (notebook 03):  Best Dice 0.6570 at epoch 15")
print(f"Improved (this notebook): Best Dice 0.6863 at epoch 40")
print(f"  -> +2.93 percentage points improvement")
print()
print(f"Dice plateaued around epoch 35-45 (0.673-0.686 range)")
print(f"  -> confirms the model reached its practical ceiling with this")
print(f"     architecture, patch size, and learning rate")
print()
print(f"W&B runs:")
print(f"  Baseline: https://wandb.ai/foroughm423/brain-tumor-segmentation-3d/runs/vqfeqgtc")
print(f"  Improved: https://wandb.ai/foroughm423/brain-tumor-segmentation-3d/runs/2mniuy59")
print()
print("Figures saved:")
for fname in ["baseline_vs_improved_comparison.png"]:
    path = os.path.join(PATHS["figures"], fname)
    status = "OK" if os.path.exists(path) else "MISSING"
    print(f"  [{status}] {fname}")
print()
print("Next -> 05_final_evaluation.ipynb")
print("  - Full test set evaluation with the improved model")
print("  - 3D volume rendering for deployment/LinkedIn visuals")
print("  - Final metrics summary for README")